# 02 - Application regression checks

This notebook reruns application-level Yamada examples against the historical reference branch and the current checkout, then compares the resulting records exactly.

It checks graph diagnostics, selected projection information where relevant, PD codes, and exact Yamada polynomial strings. Use this when you want to confirm that optimization or refactoring did not change public application results.

The reference branch is intentionally isolated to this regression notebook and its helper code. Ordinary tutorials and production workflows should not depend on it.


## 1. Prepare the comparison

The setup cell locates the repository root, creates temporary worktrees as needed, and runs the helper that records application outputs. The comparison is strict because these examples are meant to catch even small behavioral changes.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'knotted_graph').exists():
    raise RuntimeError('Run this notebook from inside the KnottedGraph checkout.')
DRIVER = ROOT / 'dev' / 'application_yamada_regression.py'
REF_RUNNER = ROOT / 'dev' / 'run_application_yamada_regression_ref.py'

import knotted_graph
from knotted_graph.invariants.yamada.native import native_available, native_import_error
from knotted_graph.invariants.yamada.factorized_frontier import native_factorized_available
print('Current notebook environment:', sys.executable)
print('Current KnottedGraph:', Path(knotted_graph.__file__).resolve())
print('Native resolved-graph Yamada backend:', native_available())
print('Factorized diagram Yamada backend:', native_factorized_available())
print('Native import error:', native_import_error())
assert native_available(), native_import_error()
assert native_factorized_available()
print('NOTE: this notebook intentionally executes detached source worktrees for correctness comparison only; it is not a performance benchmark.')

subprocess.run(['git', 'fetch', 'origin', 'Latest_Workplace'], cwd=ROOT, check=True, capture_output=True, text=True)

def run_ref(ref):
    with tempfile.TemporaryDirectory() as td:
        work = Path(td) / 'repo'
        add = subprocess.run(['git', 'worktree', 'add', '--detach', str(work), ref], cwd=ROOT, text=True, capture_output=True)
        if add.returncode:
            raise RuntimeError(f'Could not create worktree for {ref}:\n{add.stdout}\n{add.stderr}')
        try:
            env = dict(os.environ)
            env.pop('PYTHONPATH', None)
            env['PYTHONNOUSERSITE'] = '1'
            env['KG_REGRESSION_SRC'] = str(work / 'src')
            env['KG_REGRESSION_DRIVER'] = str(DRIVER)
            proc = subprocess.run([sys.executable, str(REF_RUNNER)], cwd=work, env=env, text=True, capture_output=True)
            if proc.returncode:
                raise RuntimeError(f'Application regression failed for {ref}.\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}')
            return json.loads(proc.stdout)
        finally:
            subprocess.run(['git', 'worktree', 'remove', '--force', str(work)], cwd=ROOT, text=True, capture_output=True, check=False)

baseline = run_ref('origin/Latest_Workplace')
optimized = run_ref('HEAD')
print('baseline records =', len(baseline))
print('optimized records =', len(optimized))


Current notebook environment: /Users/hakanakgun/.venvs/nh-kg/bin/python
Current KnottedGraph: /Users/hakanakgun/Desktop/Projects/ProfLeeProjects/Knotted_graph_code_paper/KnottedGraph_1earlier/src/knotted_graph/__init__.py
Native resolved-graph Yamada backend: True
Factorized diagram Yamada backend: True
Native import error: None
NOTE: this notebook intentionally executes detached source worktrees for correctness comparison only; it is not a performance benchmark.
baseline records = 38
optimized records = 38


## 2. Compare the records

This cell reports the first differences if the historical and current outputs diverge. When it finishes silently or prints a pass message, the current checkout reproduced the reference records exactly.


In [2]:
if baseline != optimized:
    differences = []
    for index, (old, new) in enumerate(zip(baseline, optimized)):
        if old != new:
            differences.append((index, old, new))
    if len(baseline) != len(optimized):
        print('record counts differ:', len(baseline), len(optimized))
    for index, old, new in differences[:10]:
        print('\nDIFFERENCE', index)
        print('Latest_Workplace:', json.dumps(old, indent=2, sort_keys=True))
        print('YAMADA_Optimization_LATEST:', json.dumps(new, indent=2, sort_keys=True))
    raise AssertionError('Application-level Yamada output changed relative to Latest_Workplace.')

print('PASS: every reproduced application Yamada record is exactly unchanged relative to Latest_Workplace.')


PASS: every reproduced application Yamada record is exactly unchanged relative to Latest_Workplace.


## 3. Summarize the tested cases

The summary prints the physics and mathematics cases included in the regression set so readers can see what the equality check covered.


In [3]:
physics = [row for row in optimized if row['application'] == 'physics']
mathematics = [row for row in optimized if row['application'] == 'mathematics']
print(f'Physics cases: {len(physics)}')
for row in physics:
    print(f"{row['case']:14s} gamma={row['gamma']:<4} V={row['nodes']:<3} E={row['edges']:<3} crossings={row['crossings']:<2} Yamada={row['yamada']}")
print(f'\nMathematics cases: {len(mathematics)}')
for row in mathematics:
    print(row['case'], row.get('yamada', row.get('yamada_negami')))


Physics cases: 12
Hopf link      gamma=0.1  V=2   E=2   crossings=2  Yamada=Y**8 + Y**7 + Y**6 + Y**5 + Y**4 + Y**3 + Y**2 + Y + 1
Hopf link      gamma=0.2  V=4   E=6   crossings=2  Yamada=Y**10 + Y**9 + Y**8 + 2*Y**7 + 2*Y**5 + 2*Y**3 + Y**2 + Y + 1
Hopf link      gamma=0.5  V=2   E=3   crossings=1  Yamada=-Y**4 - Y**3 - 2*Y**2 - Y - 1
Trefoil        gamma=0.1  V=1   E=1   crossings=3  Yamada=Y**11 + Y**10 + Y**9 + Y**8 + Y**7 - Y**4 - Y**3 - Y**2 + 1
Trefoil        gamma=0.19 V=4   E=6   crossings=3  Yamada=Y**13 - Y**9 - 3*Y**8 - Y**7 - 4*Y**6 - 2*Y**4 + Y**3 + Y**2 + 2
Trefoil        gamma=0.25 V=3   E=5   crossings=3  Yamada=-Y**6 - Y**5 - 3*Y**4 - 2*Y**3 - 3*Y**2 - Y - 1
Torus (1,2)    gamma=0.12 V=1   E=1   crossings=0  Yamada=-Y**2 - Y - 1
Torus (1,2)    gamma=0.5  V=6   E=9   crossings=2  Yamada=-Y**8 - Y**7 - 4*Y**6 - 3*Y**5 - 6*Y**4 - 3*Y**3 - 4*Y**2 - Y - 1
Torus (1,2)    gamma=0.7  V=2   E=3   crossings=1  Yamada=-Y**4 - Y**3 - 2*Y**2 - Y - 1
Solomon        gamma=0.12 V=2 

## Interpretation

A pass means the current optimized implementation reproduces the `Latest_Workplace` application-level Yamada outputs exactly for all regression cases. The historical branch dependency is intentional and isolated to this correctness-regression notebook/helper.
